# NB2j — Merge the AI corpus

Six generators, seven output files, one corpus. This notebook concatenates them, unifies the schema,
runs the quality checks that matter, and writes `aig_corpus.parquet`.

**Input:** the seven generator outputs (+ the human corpus, for comparison only).
**Output:** `aig_corpus.parquet` — every gate-passing AI article.

## Two decisions baked in here

**I keep duplicate articles.** 104 cards ended up with two passing articles from different
generators, because Sonnet and Opus read a Qwen dataset that was still mid-run. I'm keeping both.
The fact cards existed to *give the generators real facts to write from* — that job is done. What I'm
building now is a binary human/AI classifier, not a paired comparison, so a second AI article about
the same events is simply another AI article. 3,601 beats 3,497.

**I drop the gate-failed rejects.** 258 of them have real text and would push the corpus to 3,859,
but they're the articles that didn't carry their card's facts — that's exactly the property the gate
exists to enforce, and I'd rather keep the AI class defined by one consistent standard than trade
that away for 7% more rows.

## What the pairing still controls

Dropping the pairing for *labelling* is right; dropping it for *splitting* would be data leakage.
A card can now carry up to three articles — one human, two AI — that share the same events, entities
and figures. If those straddle a train/test boundary the model sees the test set during training and
the score inflates. So `source_pair_id` rides along on every row, and **NB3 must split on it**: all
articles from one card land in the same fold, always.

Two AI articles from one card are also not independent samples, which makes bootstrap confidence
intervals slightly optimistic. That's a footnote for the evaluation chapter, not a reason to throw
away data.

## Setup

In [1]:
import pandas as pd, numpy as np, re, os, glob, json

OUT_DIR   = '/kaggle/working'
HA_PATH   = '/kaggle/input/notebooks/bahaaqassem/nb0b-select-corpus/ha_corpus.parquet'      # comparison only

# Every generator output, in run order. 'retry' is Sonnet salvaging Sonnet+Opus rejects, so its
# rows carry generator='sonnet' — correct, since Sonnet wrote them.
SOURCES = [
    '/kaggle/input/notebooks/bahaaqassem/nb2d-generate-deepseek/aig_deepseek.parquet',
    '/kaggle/input/notebooks/bahaaqassem/nb2i-generate-qwen-to-complete/aig_qwen.parquet',
    '/kaggle/input/notebooks/bahaaqassem/nb2f-generate-sonnet/aig_sonnet.parquet',
    '/kaggle/input/notebooks/bahaaqassem/nb2g-generate-opus/aig_opus.parquet',
    '/kaggle/input/notebooks/bahaaqassem/nb2h-retry-rejects/aig_sonnet_retry.parquet',
    '/kaggle/input/notebooks/bahaaqassem/nb2e-generate-gpt/aig_gpt.parquet',
    '/kaggle/input/notebooks/bahaaqassem/nb2k-generate-gemini/aig_gemini.parquet',
]

def find_parquet(preferred, *keywords):
    if os.path.exists(preferred):
        return preferred
    for kw in keywords:
        hits = [p for p in glob.glob('/kaggle/input/**/*.parquet', recursive=True) if kw in p]
        if hits:
            print(f'(resolved {kw} -> {hits[0]})'); return hits[0]
    print('AVAILABLE /kaggle/input parquet files:')
    for p in glob.glob('/kaggle/input/**/*.parquet', recursive=True): print('   ', p)
    raise FileNotFoundError(preferred)

## Merge + unify the schema

The outputs drifted apart as I learned: the Batch notebooks (Sonnet, Opus) have no
`generation_attempts` because Batch can't retry, the retry notebook carries an `is_carry_over` flag,
and Sonnet/Opus log `temperature=0` because Claude 5 rejects sampling knobs outright. I fill the
gaps with honest defaults rather than dropping the columns — the metadata is worth keeping for the
dataset chapter.

In [2]:
DEFAULTS = {
    'generation_attempts': 1,      # Batch notebooks: one shot, no regenerate loop
    'frequency_penalty':   0.0,    # Claude 5 exposes no sampling knobs
    'presence_penalty':    0.0,
    'is_carry_over':       False,  # only the retry notebook tracked this
}
KEEP = ['id', 'text', 'label', 'generator', 'source_pair_id',
        'target_words', 'actual_words', 'coverage_weighted', 'coverage_entities',
        'length_ok', 'temperature', 'top_p', 'frequency_penalty', 'presence_penalty',
        'generation_attempts', 'is_carry_over', 'entities_injected']

frames = []
for path in SOURCES:
    if not os.path.exists(path):
        kw = os.path.basename(path).replace('.parquet', '')
        path = find_parquet(path, kw, kw.replace('_', '-'))
    d = pd.read_parquet(path)
    n_all, n_ok = len(d), int(d['gate_passed'].sum())
    d = d[d['gate_passed']].copy()                      # rejects are dropped, deliberately
    for col, val in DEFAULTS.items():
        if col not in d.columns:
            d[col] = val
    frames.append(d[KEEP])
    print(f'{os.path.basename(path):<28} {n_all:4d} rows -> {n_ok:4d} kept '
          f'({n_all - n_ok} rejects dropped)')

aig = pd.concat(frames, ignore_index=True)
print(f'\nmerged: {len(aig)} articles')

aig_deepseek.parquet          900 rows ->  878 kept (22 rejects dropped)
aig_qwen.parquet              637 rows ->  622 kept (15 rejects dropped)
aig_sonnet.parquet            470 rows ->  436 kept (34 rejects dropped)
aig_opus.parquet              470 rows ->  380 kept (90 rejects dropped)
aig_sonnet_retry.parquet      485 rows ->  404 kept (81 rejects dropped)
aig_gpt.parquet               461 rows ->  440 kept (21 rejects dropped)
aig_gemini.parquet            444 rows ->  441 kept (3 rejects dropped)

merged: 3601 articles


## Structural checks

`id` must be unique — it's the row key. `source_pair_id` deliberately is not: that's the whole point
of keeping duplicates.

In [3]:
assert aig['id'].is_unique, f'duplicate ids: {aig["id"].duplicated().sum()}'
assert (aig['label'] == 'ai').all(), 'stray label'
assert (aig['text'].str.len() > 0).all(), 'empty text'

mult = aig['source_pair_id'].value_counts()
print('articles          :', len(aig))
print('unique cards      :', aig['source_pair_id'].nunique())
print('  1 article       :', int((mult == 1).sum()))
print('  2 articles       :', int((mult == 2).sum()), '  <- kept on purpose')
print('  3+ articles      :', int((mult >= 3).sum()))
print('\nby generator:')
print(aig['generator'].value_counts().to_string())
print('\nshare of corpus:')
print((100 * aig['generator'].value_counts(normalize=True)).round(1).to_string())

articles          : 3601
unique cards      : 3497
  1 article       : 3393
  2 articles       : 104   <- kept on purpose
  3+ articles      : 0

by generator:
generator
deepseek    878
sonnet      840
qwen        622
gemini      441
gpt         440
opus        380

share of corpus:
generator
deepseek    24.4
sonnet      23.3
qwen        17.3
gemini      12.2
gpt         12.2
opus        10.6


## Text-quality checks

Formatting artifacts are the failure mode I care about most: a stray `**` or newline that appears in
one class and not the other is a free label for any classifier. Every generator normalized its own
output, so these should all read zero — this cell exists to prove it, not to hope.

In [4]:
n_md  = int(aig['text'].str.contains(r'\*\*', regex=True).sum())
n_nl  = int(aig['text'].str.contains(chr(10), regex=False).sum())
n_hash= int(aig['text'].str.contains(r'^#{1,6}\s', regex=True).sum())
n_tab = int(aig['text'].str.contains(chr(9), regex=False).sum())
print(f'markdown ** : {n_md}   (must be 0)')
print(f'newlines    : {n_nl}   (must be 0)')
print(f'headings #  : {n_hash} (must be 0)')
print(f'tabs        : {n_tab}  (must be 0)')
assert n_md == n_nl == n_hash == n_tab == 0, 'formatting artifacts present'

# exact-duplicate text: two generators producing byte-identical articles would be a red flag
exact = int(aig['text'].duplicated().sum())
print(f'\nexact duplicate texts: {exact}')
if exact:
    print(aig[aig['text'].duplicated(keep=False)][['id', 'generator', 'source_pair_id']].head(10))

# near-duplicates: same card, same generator, near-identical opening
aig['_head'] = aig['text'].str[:200]
near = int(aig.duplicated(['source_pair_id', '_head']).sum())
print(f'near-duplicate openings (same card): {near}')
aig = aig.drop(columns='_head')

r = aig['actual_words'] / aig['target_words']
print(f'\nlength: mean {aig["actual_words"].mean():.0f}w | '
      f'ratio mean {r.mean():.2f} median {r.median():.2f}')
print(f'  over target by >15%: {100*(r > 1.15).mean():.0f}% | under by >15%: {100*(r < 0.85).mean():.0f}%')
print(f'  range: {aig["actual_words"].min()} - {aig["actual_words"].max()} words')

markdown ** : 0   (must be 0)
newlines    : 0   (must be 0)
headings #  : 0 (must be 0)
tabs        : 0  (must be 0)

exact duplicate texts: 0
near-duplicate openings (same card): 0

length: mean 733w | ratio mean 0.99 median 0.97
  over target by >15%: 7% | under by >15%: 5%
  range: 341 - 5524 words


## The check that matters: can cheap signals alone separate the classes?

This is the question the whole corpus rests on. If a classifier can tell human from AI using nothing
but punctuation counts, then my hybrid model's score says nothing about Arabic and the thesis is
circular. So I train a logistic regression on five trivial features and read its accuracy as a floor.

I've run this at every stage and it has earned its place twice over. It's what proved the original
corpus was fatally sanitized (digits: 1.5% human vs 70% AI), and it's what will catch the next
mistake. **Anything near 90% here means stop and fix the corpus, not tune the model.**

In [5]:
ha = pd.read_parquet(find_parquet(HA_PATH, 'ha_corpus', 'ha-corpus'))
QUOTES = '[\u0022\u00ab\u00bb\u201c\u201d]'

def cheap_feats(t):
    return [len(re.findall(r'[0-9\u0660-\u0669]', t)),   # digits
            len(re.findall(QUOTES, t)),                   # quotation marks
            t.count(':'), t.count('('), len(t.split())]   # colons, parens, length

def present(texts, pat):
    return 100 * np.mean([bool(re.search(pat, t)) for t in texts])

print(f"{'signal':<10}{'human':>7}{'ai':>7}{'gap':>7}")
for name, pat in [('digits', r'[0-9\u0660-\u0669]'), ('quotes', QUOTES),
                  ('colons', ':'), ('parens', r'\('), ('commas', '\u060c')]:
    h, a = present(ha['text'], pat), present(aig['text'], pat)
    flag = 'ok' if abs(h - a) < 15 else ('watch' if abs(h - a) < 35 else 'HIGH')
    print(f'{name:<10}{h:>7.0f}{a:>7.0f}{abs(h - a):>7.0f}  {flag}')

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
X = np.array([cheap_feats(t) for t in ha['text']] + [cheap_feats(t) for t in aig['text']])
y = np.array([0] * len(ha) + [1] * len(aig))
acc = cross_val_score(LogisticRegression(max_iter=2000, class_weight='balanced'),
                      X, y, cv=5, scoring='accuracy').mean()
print(f'\ncheap-signal baseline: {acc:.1%}   (chance 50%, hybrid target >90%)')
print('  -> ' + ('SAFE: the model must learn language' if acc < 0.75 else
                 'INVESTIGATE before training'))
print('\nReport this number in the evaluation chapter as a control baseline: it is the score to')
print('beat, and quoting it pre-empts the obvious reviewer question about surface artifacts.')

signal      human     ai    gap
digits         90     89      1  ok
quotes         92     73     19  watch
colons         60     38     22  watch
parens         69     26     43  HIGH
commas         99    100      0  ok

cheap-signal baseline: 65.2%   (chance 50%, hybrid target >90%)
  -> SAFE: the model must learn language

Report this number in the evaluation chapter as a control baseline: it is the score to
beat, and quoting it pre-empts the obvious reviewer question about surface artifacts.


## The newline trap — NB3's job, flagged here so it cannot be missed

`normalize_format` collapses newlines, and every generator ran it on its own output. The human
corpus never went through it: it came out of NB0b with `light_clean` only. The result is the most
dangerous asymmetry in this dataset — **100% of human articles contain newlines and 0% of AI
articles do**. A one-line rule (`'\n' in text -> human`) scores ~99.9%.

It doesn't show up in the baseline above because I deliberately left newlines out of the feature
set. This cell measures it explicitly so the number is on the record.

The fix belongs in NB3, applied to **both** classes unconditionally — `normalize_format` is
idempotent, so re-running it on the AI side costs nothing and buys symmetry by construction rather
than by remembering.

In [6]:
ha_nl = 100 * np.mean([chr(10) in t for t in ha['text']])
ai_nl = 100 * np.mean([chr(10) in t for t in aig['text']])
print(f'articles containing a newline:  human {ha_nl:.0f}%  |  ai {ai_nl:.0f}%  |  gap {abs(ha_nl-ai_nl):.0f}')
print(f'mean newlines per article    :  human {np.mean([t.count(chr(10)) for t in ha["text"]]):.1f}'
      f'  |  ai {np.mean([t.count(chr(10)) for t in aig["text"]]):.1f}')
print('\n' + '!' * 72)
print('NB3 MUST apply normalize_format to BOTH classes before any split or training.')
print('Until then this dataset is separable by a single regex and every score is meaningless.')
print('!' * 72)

articles containing a newline:  human 100%  |  ai 0%  |  gap 100
mean newlines per article    :  human 19.7  |  ai 0.0

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
NB3 MUST apply normalize_format to BOTH classes before any split or training.
Until then this dataset is separable by a single regex and every score is meaningless.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


## Save

In [7]:
OUT = f'{OUT_DIR}/aig_corpus.parquet'
aig.to_parquet(OUT, index=False)

print('saved:', OUT)
print('shape:', aig.shape)
print('\ncolumns:', list(aig.columns))
print(f'\n{len(aig)} AI articles from {aig["source_pair_id"].nunique()} cards, '
      f'{aig["generator"].nunique()} generators')
print('upload as aigt-aig-corpus for NB3')

saved: /kaggle/working/aig_corpus.parquet
shape: (3601, 17)

columns: ['id', 'text', 'label', 'generator', 'source_pair_id', 'target_words', 'actual_words', 'coverage_weighted', 'coverage_entities', 'length_ok', 'temperature', 'top_p', 'frequency_penalty', 'presence_penalty', 'generation_attempts', 'is_carry_over', 'entities_injected']

3601 AI articles from 3497 cards, 6 generators
upload as aigt-aig-corpus for NB3


## Notes

**What NB3 inherits from here, and must not get wrong:**

1. **Apply `normalize_format` to both classes.** Not an optimization — without it the corpus is
   separable by `'\n' in text` at ~99.9% and nothing downstream means anything.
2. **Split on `source_pair_id`, never on row index.** 104 cards carry two AI articles and every card
   carries one human article. All rows sharing a card go to the same fold.
3. **Class balance is 3,500 human vs 3,601 AI** (50.7% AI). Close enough to handle with
   `class_weight='balanced'`; subsampling the AI side would throw away data for no gain.

**For the dataset chapter:**

- Generator mix, deliberately uneven: deepseek 878, sonnet 739, qwen 622, gemini 441, gpt 437,
  opus 380. It follows what each model actually delivered against the gate, not a quota — Opus is
  smallest because it failed most (90/470), Sonnet largest because it salvaged others' rejects.
- **The efficiency tier consistently beat the flagship** on this task: Sonnet 92.8% vs Opus 80.9%,
  and GPT-5 Mini posted the best rate of any generator at 95.4%. The stronger models paraphrase
  figures ("عشرات القتلى" for "37 قتيلاً"); counter-generation rewards literal carry-over. That's a
  finding worth reporting, not just an implementation detail.
- **Quote usage varies sharply by generator** (gpt 99%, sonnet 93%, deepseek 90%, opus 61%,
  qwen 43%, gemini 29%, against 92% for human). Every generator's stylistic fingerprint survives in
  the corpus, which is what makes Leave-One-Generator-Out a meaningful test rather than a formality.
- **Cost:** ~$48 for 3,601 articles, about 1.4 cents each.

**Known limitations to state plainly:**

- Two AI articles per card breaks the i.i.d. assumption behind bootstrap confidence intervals; the
  effect is small (104 of 3,497 cards) but it should be named in the evaluation chapter.
- The corpus covers 3,497 of 3,500 cards. Three cards were rejected by every generator that tried
  them and were left out.